# VoxSentinel AI — Synthetic Speech Detection fine-tune
Prepared configuration for VoxSentinel AI. **No model is trained by the project by default.**
Run top-to-bottom in Colab (T4/A100). Every section prints what it actually did.


## 1. Environment
Install pinned deps, verify GPU.

In [ ]:
!pip -q install "transformers==4.44.*" "datasets==2.20.*" "librosa==0.10.*" soundfile torchaudio scikit-learn
import torch, platform
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), platform.platform())

## 2. Data
ASVspoof 2019 LA is access-by-request (asvspoof.org). Mount Drive if you already have it.
Do **not** redistribute the audio.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/asvspoof2019/LA'
import os
print(sorted(os.listdir(DATA))[:8] if os.path.isdir(DATA) else "ASVspoof not found — request access first")

## 3. Manifest + leakage guard
Reuse the project's deterministic splitter so splits match the repo.

In [ ]:
!git clone -q https://github.com/<your-org>/voxsentinel-ai.git 2>/dev/null || true
# If the repo is private, upload training/datasets/manifest.py instead.
!python voxsentinel-ai/training/datasets/manifest.py librispeech --root /content/librispeech 2>/dev/null || true

## 4. Model + training

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer
import torch
MODEL = "facebook/wav2vec2-base"          # baseline; swap for aasist-style head if GPU >= 16GB
fe = AutoFeatureExtractor.from_pretrained(MODEL)
model = AutoModelForAudioClassification.from_pretrained(MODEL, num_labels=2,
        id2label={0: "bonafide", 1: "spoof"}, label2id={"bonafide": 0, "spoof": 1})
args = TrainingArguments(output_dir="out", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=3e-4, num_train_epochs=6, warmup_ratio=0.1, lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(), eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="eer", greater_is_better=False,
    report_to="none", dataloader_num_workers=2)
print(args)

## 5. Metrics
EER is the primary metric (threshold-free). Report per quality bucket (clean/degraded/insufficient) because the project gates on audio quality.

In [ ]:
import numpy as np
def eer(scores, labels):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    return {"eer": eer(probs, p.label_ids)}
print("hook ready: pass compute_metrics=compute_metrics to Trainer")

## 6. Export + integration

In [ ]:
# model.save_pretrained("voxsentinel-detector"); fe.save_pretrained("voxsentinel-detector")
# Then copy the folder to backend/models/ and set DETECTOR_MODEL_DIR.
# IMPORTANT: the app's threshold policy (app/services/risk_engine.py) must not be changed
# to make scores look better. Re-run tests/test_audio_quality.py afterwards.
print("export instructions above are documentation, not executed training")